# NSL-KDD Intrusion Detection

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, pairwise_distances_argmin
from sklearn.metrics import pairwise_distances_argmin_min

from imblearn.over_sampling import BorderlineSMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import shap
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
train = pd.read_csv("kdd_train.csv", header=None)
test = pd.read_csv("kdd_test.csv", header=None)

columns = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes","land",
    "wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate",
    "label"
]

train.columns = columns
test.columns = columns

train = train.iloc[1:].copy()
test = test.iloc[1:].copy()

## 2. Label Mapping & Filtering

In [ ]:
common_labels = set(train['label']) & set(test['label'])
train = train[train['label'].isin(common_labels)]
test = test[test['label'].isin(common_labels)]

def map_attack(label):
    if label == 'normal':
        return 'normal'
    elif label in ['back','land','neptune','pod','smurf','teardrop',
                   'apache2','udpstorm','processtable','worm']:
        return 'dos'
    elif label in ['ipsweep','nmap','portsweep','satan','mscan','saint']:
        return 'probe'
    elif label in ['ftp_write','guess_passwd','imap','multihop','phf','spy',
                   'warezclient','warezmaster','sendmail','named',
                   'snmpgetattack','snmpguess','xlock','xsnoop','httptunnel']:
        return 'r2l'
    elif label in ['buffer_overflow','loadmodule','perl','rootkit',
                   'ps','sqlattack','xterm']:
        return 'u2r'
    else:
        return 'unknown'

train['label'] = train['label'].apply(map_attack)
test['label'] = test['label'].apply(map_attack)

X_train = train.drop(columns=["label"])
y_train = train["label"]
X_test = test.drop(columns=["label"])
y_test = test["label"]

print("Train distribution:", Counter(y_train))
print("Test distribution:", Counter(y_test))

Train distribution: Counter({'normal': 67343, 'dos': 45927, 'probe': 11656, 'r2l': 993, 'u2r': 52})
Test distribution: Counter({'normal': 11245, 'dos': 7562, 'probe': 1754, 'r2l': 811, 'u2r': 26})


## 3. Preprocessing
### FIX: LabelEncoder disimpan per kolom dalam dict agar tidak tertimpa

In [ ]:
categorical_cols = ["protocol_type", "service", "flag"]

# FIX: simpan setiap encoder dalam dict, bukan variabel tunggal yang ditimpa
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    # Handle unseen categories di test set
    X_test[col] = X_test[col].apply(
        lambda x: x if x in le.classes_ else le.classes_[0]
    )
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le  # simpan encoder per kolom

print("Categorical encoding done. Encoders saved:", list(label_encoders.keys()))

Categorical encoding done. Encoders saved: ['protocol_type', 'service', 'flag']


## 4. Target Label Encoder & Scaling (Full Features)

In [ ]:
# Label encoder untuk target - fit hanya dari y_train
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train_encoded = label_encoder.transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Scaling - fit hanya dari X_train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Classes:", label_encoder.classes_)
print("X_train_scaled shape:", X_train_scaled.shape)

Classes: ['dos' 'normal' 'probe' 'r2l' 'u2r']
X_train_scaled shape: (125971, 41)


## 5. Baseline: XGBoost tanpa Sampling (Unbalanced)

In [ ]:
print("Training baseline XGBoost (unbalanced)...")

model_xgb_unbalanced = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
model_xgb_unbalanced.fit(X_train_scaled, y_train_encoded)
y_pred_unbalanced = model_xgb_unbalanced.predict(X_test_scaled)

print("\n=== Baseline (Unbalanced) ===")
print(classification_report(y_test_encoded, y_pred_unbalanced,
                             target_names=label_encoder.classes_))

Training baseline XGBoost (unbalanced)...

=== Baseline (Unbalanced) ===
              precision    recall  f1-score   support

         dos       1.00      1.00      1.00      7562
      normal       0.94      0.99      0.97     11245
       probe       0.96      1.00      0.98      1754
         r2l       1.00      0.20      0.33       811
         u2r       1.00      0.46      0.63        26

    accuracy                           0.96     21398
   macro avg       0.98      0.73      0.78     21398
weighted avg       0.97      0.96      0.95     21398



## 6. Hybrid Sampling: KMeans Undersampling + BorderlineSMOTE

In [ ]:
def kmeans_undersample(
    X,
    y,
    sampling_strategy,
    cluster_strategy,
    random_state=42
):

    X_result, y_result = [], []

    unique_classes = np.unique(y)

    rng = np.random.RandomState(random_state)

    for cls in unique_classes:

        # =====================================================
        # AMBIL DATA PER CLASS
        # =====================================================
        mask = (y == cls) if not hasattr(y, 'values') else (y.values == cls)

        X_cls = X[mask]

        # target jumlah setelah undersampling
        target_n = sampling_strategy.get(cls, len(X_cls))

        # jumlah cluster untuk tiap class
        n_cluster_cls = cluster_strategy.get(cls, 40)

        # =====================================================
        # MINORITY CLASS -> AMBIL SEMUA
        # =====================================================
        if len(X_cls) <= target_n:

            X_result.append(X_cls)
            y_result.extend([cls] * len(X_cls))

        # =====================================================
        # MAJORITY CLASS -> KMEANS CLUSTER SAMPLING
        # =====================================================
        else:

            print(f"\nClass : {cls}")
            print(f"Original samples : {len(X_cls)}")
            print(f"Target samples   : {target_n}")
            print(f"Clusters used    : {n_cluster_cls}")

            km = KMeans(
                n_clusters=n_cluster_cls,
                random_state=random_state,
                n_init=10,
                max_iter=300
            )

            cluster_labels = km.fit_predict(X_cls)

            # jumlah sampel yang diambil dari tiap cluster
            samples_per_cluster = target_n // n_cluster_cls

            selected_indices = []

            # =====================================================
            # SAMPLING TIAP CLUSTER
            # =====================================================
            for cluster_id in range(n_cluster_cls):

                cluster_idx = np.where(cluster_labels == cluster_id)[0]

                if len(cluster_idx) <= samples_per_cluster:

                    selected_indices.extend(cluster_idx)

                else:

                    chosen = rng.choice(
                        cluster_idx,
                        size=samples_per_cluster,
                        replace=False
                    )

                    selected_indices.extend(chosen)

            selected_indices = np.array(selected_indices)

            # =====================================================
            # JIKA JUMLAH MASIH KURANG
            # =====================================================
            if len(selected_indices) < target_n:

                remaining = np.setdiff1d(
                    np.arange(len(X_cls)),
                    selected_indices
                )

                extra = rng.choice(
                    remaining,
                    size=target_n - len(selected_indices),
                    replace=False
                )

                selected_indices = np.concatenate([
                    selected_indices,
                    extra
                ])

            X_result.append(X_cls[selected_indices])

            y_result.extend([cls] * len(selected_indices))

    # =====================================================
    # GABUNGKAN SEMUA CLASS
    # =====================================================
    X_out = np.vstack(X_result)
    y_out = np.array(y_result)

    return X_out, y_out

In [ ]:
from collections import Counter
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import BorderlineSMOTE

# =========================================================
# PARAMETER
# =========================================================

sampling_strategy_under = {
    'normal': 15000,
    'dos': 10000,
    'probe': 8000,
}

# jumlah cluster untuk setiap kelas mayoritas
cluster_strategy = {
    'normal': 40,
    'dos': 40,
    'probe': 40,
}

sampling_strategy_smote = {
    'r2l': 6000,
    'u2r': 5000,
}

experiments = {
    "Hybrid Sampling": "hybrid",
    "KMeans Only": "kmeans",
    "BorderlineSMOTE Only": "smote"
}

# =========================================================
# LOOP EKSPERIMEN
# =========================================================

for exp_name, exp_type in experiments.items():

    print("=" * 70)
    print(exp_name)
    print("=" * 70)

    # =====================================================
    # SAMPLING
    # =====================================================

    if exp_type == "hybrid":

        # Step 1 : KMeans Undersampling
        X_sample, y_sample = kmeans_undersample(
            X_train_scaled,
            y_train,
            sampling_strategy=sampling_strategy_under,
            cluster_strategy=cluster_strategy,
            random_state=42
        )

        # Step 2 : BorderlineSMOTE
        smote = BorderlineSMOTE(
            kind='borderline-2',
            sampling_strategy=sampling_strategy_smote,
            k_neighbors=2,
            random_state=42
        )

        X_sample, y_sample = smote.fit_resample(
            X_sample,
            y_sample
        )

    elif exp_type == "kmeans":

        X_sample, y_sample = kmeans_undersample(
            X_train_scaled,
            y_train,
            sampling_strategy=sampling_strategy_under,
            cluster_strategy=cluster_strategy,
            random_state=42
        )

    elif exp_type == "smote":

        smote = BorderlineSMOTE(
            kind='borderline-2',
            sampling_strategy=sampling_strategy_smote,
            k_neighbors=2,
            random_state=42
        )

        X_sample, y_sample = smote.fit_resample(
            X_train_scaled,
            y_train
        )

    # =====================================================
    # CEK DISTRIBUSI KELAS
    # =====================================================

    print("\nClass Distribution:")
    print(Counter(y_sample))

    # =====================================================
    # ENCODE LABEL
    # =====================================================

    y_sample_encoded = label_encoder.transform(y_sample)

    # =====================================================
    # TRAIN XGBOOST
    # =====================================================

    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_sample, y_sample_encoded)

    # =====================================================
    # PREDIKSI
    # =====================================================

    y_pred = model.predict(X_test_scaled)

    # =====================================================
    # HASIL
    # =====================================================

    print("\nClassification Report:")
    print(
        classification_report(
            y_test_encoded,
            y_pred,
            target_names=label_encoder.classes_
        )
    )

Hybrid Sampling

Class : dos
Original samples : 45927
Target samples   : 10000
Clusters used    : 40

Class : normal
Original samples : 67343
Target samples   : 15000
Clusters used    : 40

Class : probe
Original samples : 11656
Target samples   : 8000
Clusters used    : 40

Class Distribution:
Counter({np.str_('normal'): 15000, np.str_('dos'): 10000, np.str_('probe'): 8000, np.str_('r2l'): 6000, np.str_('u2r'): 5000})

Classification Report:
              precision    recall  f1-score   support

         dos       1.00      1.00      1.00      7562
      normal       0.95      0.99      0.97     11245
       probe       0.96      1.00      0.98      1754
         r2l       0.99      0.26      0.41       811
         u2r       0.89      0.62      0.73        26

    accuracy                           0.97     21398
   macro avg       0.96      0.77      0.82     21398
weighted avg       0.97      0.97      0.96     21398

KMeans Only

Class : dos
Original samples : 45927
Target samples